<img src="https://raw.githubusercontent.com/Delarbol/semillero/master/Images-DS/alt_banner.png" width=100%>

<h1 style="margin:0">Inteligencia Artificial Aplicada a la Economía II</h1>
<h2 style="margin:.35em 0 0 0">NLP y texto como datos</h2>
<p style="margin:.8em 0 0 0"><strong>Sesiones:</strong> lunes 14 y viernes 18 de septiembre de 2026 · 7:00–9:00 a. m.</p>
<p style="margin:.3em 0 0 0"><strong>Evidencia:</strong> Taller 4</p>

> **Nota sobre los datos.** Los textos de este cuaderno son simulados y fueron creados con fines pedagógicos. No describen hechos reales ni deben citarse como evidencia económica.

# Propósito

Este cuaderno presenta el texto como una fuente de datos. El objetivo no es acumular técnicas de NLP, sino aprender una secuencia defendible:

1. formular una pregunta económica;
2. definir la unidad de análisis y las etiquetas;
3. representar el texto numéricamente;
4. entrenar y validar un modelo;
5. interpretar sentimiento y temas con prudencia;
6. volver a los documentos para verificar cada afirmación importante.

La idea central es sencilla: **un modelo organiza evidencia textual; no reemplaza la lectura ni convierte asociaciones en hechos**.

## Resultados de aprendizaje

Al finalizar las dos sesiones, usted debería poder:

- explicar qué información preservan y qué información pierden *bag of words*, TF–IDF y un *embedding* denso;
- construir un `Pipeline` reproducible para clasificar textos;
- comparar el modelo contra un *baseline* y evaluarlo con validación cruzada, matriz de confusión y F1 macro;
- diferenciar clasificación supervisada de descubrimiento no supervisado de temas;
- evaluar sentimiento sin confundir tono, tema y posición económica;
- recuperar fragmentos de respaldo mediante similitud coseno;
- documentar qué resultados están respaldados, cuáles requieren revisión y cuáles deben descartarse.

## Ruta de las dos sesiones

| Sesión | Minutos | Actividad | Producto verificable |
|---|---:|---|---|
| **Lun. 7** | 0–15 | Pregunta económica y unidad de análisis | Variable objetivo y unidad definidas |
|  | 15–45 | De palabras a números | Matrices BoW y TF–IDF interpretadas |
|  | 45–70 | *Embeddings* y similitud | Vecinos textuales razonables |
|  | 70–110 | Clasificación, baseline y validación | Pipeline evaluado sin fuga |
|  | 110–120 | Cierre | *Exit ticket* |
| **Vie. 11** | 0–30 | Sentimiento | Baseline léxico y modelo supervisado comparados |
|  | 30–65 | Temas | Componentes NMF interpretados con cautela |
|  | 65–100 | Validación con fuentes | Matriz afirmación–evidencia |
|  | 100–120 | Lanzamiento del Taller 4 | Plan de trabajo y criterios de entrega |

**Ritmo sugerido:** antes de ejecutar una celda importante, escriba qué espera observar. Después compare la predicción con la salida real.

# 0. Preparación del entorno

El desarrollo principal funciona sin descargas externas. Requiere `pandas`, `numpy`, `matplotlib`, `seaborn` y `scikit-learn`.

In [ ]:
import re
import warnings
from textwrap import shorten

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import NMF, TruncatedSVD
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    classification_report,
    confusion_matrix,
    f1_score,
    pairwise_distances,
)
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import Normalizer

warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 100)
sns.set_theme(style='whitegrid')

RANDOM_STATE = 42

## Corpus pedagógico de comunicaciones económicas

Cada fila representa **un fragmento**, no una institución ni una fecha completa. Esta decisión define qué significa una predicción: clasificaremos fragmentos individuales.

Variables principales:

| Variable | Significado |
|---|---|
| `texto_id` | identificador único del fragmento |
| `fecha` | fecha simulada de publicación |
| `fuente_tipo` | tipo de emisor |
| `texto` | fragmento analizado |
| `tema` | etiqueta temática para la sesión guiada |
| `sentimiento` | tono económico anotado: negativo, neutral o positivo |

La función siguiente genera un corpus reproducible con vocabulario compartido entre categorías. Por eso el problema no se reduce a buscar una sola palabra.

In [ ]:
def construir_corpus(n=420, semilla=42):
    rng = np.random.default_rng(semilla)

    temas = {
        'inflación': {
            'sujetos': ['la inflación anual', 'el costo de vida', 'la variación de precios', 'la inflación básica'],
            'objetos': ['alimentos', 'servicios', 'bienes regulados', 'vivienda'],
        },
        'empleo': {
            'sujetos': ['la tasa de desempleo', 'la ocupación urbana', 'el empleo formal', 'la participación laboral'],
            'objetos': ['jóvenes', 'mujeres', 'trabajadores formales', 'zonas urbanas'],
        },
        'actividad': {
            'sujetos': ['la actividad económica', 'la producción industrial', 'el comercio minorista', 'la demanda interna'],
            'objetos': ['industria', 'servicios', 'comercio', 'construcción'],
        },
        'sector externo': {
            'sujetos': ['las exportaciones', 'la balanza comercial', 'las importaciones', 'el tipo de cambio'],
            'objetos': ['manufacturas', 'energía', 'bienes de capital', 'productos agrícolas'],
        },
        'fiscal': {
            'sujetos': ['el balance fiscal', 'el recaudo tributario', 'el gasto público', 'la deuda del gobierno'],
            'objetos': ['inversión pública', 'transferencias', 'impuestos', 'servicio de deuda'],
        },
    }

    marcos = {
        'positivo': [
            '{sujeto} mejoró durante {periodo}',
            '{sujeto} mostró una recuperación gradual en {periodo}',
            'se observó una evolución favorable de {sujeto} durante {periodo}',
            '{sujeto} avanzó más de lo previsto en {periodo}',
        ],
        'neutral': [
            '{sujeto} se mantuvo estable durante {periodo}',
            '{sujeto} registró pocos cambios en {periodo}',
            'el informe describió la evolución de {sujeto} durante {periodo}',
            '{sujeto} permaneció cerca del nivel anterior en {periodo}',
        ],
        'negativo': [
            '{sujeto} se deterioró durante {periodo}',
            '{sujeto} presentó una caída marcada en {periodo}',
            'se observó una evolución desfavorable de {sujeto} durante {periodo}',
            '{sujeto} retrocedió más de lo previsto en {periodo}',
        ],
    }

    conectores = [
        'El resultado estuvo asociado con {objeto}.',
        'El cambio fue heterogéneo y se concentró en {objeto}.',
        'La incertidumbre sigue siendo alta para {objeto}.',
        'La comparación interanual debe interpretarse con cautela en {objeto}.',
        'El documento no atribuye causalidad, pero destaca el comportamiento de {objeto}.',
    ]
    periodos = ['enero', 'febrero', 'marzo', 'el primer trimestre', 'el segundo trimestre', 'el último mes']
    fuentes = ['boletín técnico', 'informe sectorial', 'comunicado empresarial', 'nota de investigación']
    regiones = ['nacional', 'Caribe', 'Andina', 'Pacífica', 'Orinoquía']
    sentimientos = np.array(['negativo', 'neutral', 'positivo'])
    probs_sent = np.array([0.27, 0.51, 0.22])
    nombres_tema = list(temas)

    filas = []
    usados = set()
    i = 0
    while len(filas) < n:
        tema = rng.choice(nombres_tema, p=[0.23, 0.22, 0.21, 0.18, 0.16])
        sentimiento = rng.choice(sentimientos, p=probs_sent)
        sujeto = rng.choice(temas[tema]['sujetos'])
        objeto = rng.choice(temas[tema]['objetos'])
        periodo = rng.choice(periodos)
        frase = rng.choice(marcos[sentimiento]).format(sujeto=sujeto, periodo=periodo)
        detalle = rng.choice(conectores).format(objeto=objeto)
        region = rng.choice(regiones)
        contraste = ''
        if rng.random() < 0.42:
            otros_temas = [t for t in nombres_tema if t != tema]
            tema_secundario = rng.choice(otros_temas)
            sujeto_secundario = rng.choice(temas[tema_secundario]['sujetos'])
            objeto_secundario = rng.choice(temas[tema_secundario]['objetos'])
            sentimiento_secundario = rng.choice(sentimientos)
            frase_secundaria = rng.choice(marcos[sentimiento_secundario]).format(
                sujeto=sujeto_secundario,
                periodo=rng.choice(periodos),
            )
            contraste = (
                f' En contraste, {frase_secundaria.lower()} '
                f'y el análisis secundario destacó {objeto_secundario}.'
            )
        texto = f'{frase}. {detalle}{contraste} Cobertura: {region}.'
        if texto in usados:
            i += 1
            continue
        usados.add(texto)
        filas.append({
            'texto_id': f'TX-{len(filas)+1:04d}',
            'fecha': pd.Timestamp('2025-01-01') + pd.Timedelta(days=int(rng.integers(0, 560))),
            'fuente_tipo': rng.choice(fuentes),
            'texto': texto,
            'tema': tema,
            'sentimiento': sentimiento,
        })
        i += 1
        if i > n * 30:
            raise RuntimeError('No fue posible construir textos únicos.')
    return pd.DataFrame(filas).sort_values('fecha').reset_index(drop=True)


corpus = construir_corpus()
corpus.head(4)

### Antes de ejecutar la auditoría

Anote su predicción:

1. ¿Qué clase temática será la más frecuente?
2. ¿Habrá valores faltantes?
3. ¿La variable `sentimiento` estará balanceada?
4. ¿Qué problema producirían textos duplicados entre entrenamiento y prueba?

In [ ]:
auditoria = pd.Series({
    'filas': len(corpus),
    'textos_únicos': corpus['texto'].nunique(),
    'faltantes_totales': int(corpus.isna().sum().sum()),
    'fecha_mínima': corpus['fecha'].min().date(),
    'fecha_máxima': corpus['fecha'].max().date(),
    'longitud_mediana_palabras': corpus['texto'].str.split().str.len().median(),
})
display(auditoria.to_frame('valor'))
display(pd.crosstab(corpus['tema'], corpus['sentimiento'], margins=True))

# Sesión 1 · Representación, embeddings y clasificación

## 1. El texto no entra directamente a un modelo

Un algoritmo necesita números. Representar texto significa construir una función

\[
f(\text{documento}) \rightarrow \mathbf{x}\in\mathbb{R}^p.
\]

La representación decide qué semejanzas puede percibir el modelo.

| Representación | Idea | Fortalezas | Límites |
|---|---|---|---|
| Bolsa de palabras | cuenta términos | interpretable y rápida | ignora orden y contexto |
| TF–IDF | pondera términos frecuentes en un documento pero raros en el corpus | baseline fuerte y auditable | semántica limitada |
| Embedding denso | resume el documento en pocas dimensiones | similitud y estructura latente | interpretación menos directa |
| Transformer preentrenado | usa contexto aprendido en grandes corpus | captura polisemia y contexto | costo, dependencia y riesgo de desalineación |

No existe una representación universalmente mejor. La elección depende de la pregunta, el idioma, el tamaño del corpus, la trazabilidad exigida y el costo computacional.

## 2. Bolsa de palabras

`CountVectorizer` construye un vocabulario y cuenta cuántas veces aparece cada término. En este microejemplo usaremos unigramas y bigramas.

> **Prediga antes de ejecutar:** ¿qué términos permitirán separar inflación de empleo? ¿Qué palabras serán poco informativas porque aparecen en casi todos los textos?

In [ ]:
microtextos = corpus.groupby('tema', group_keys=False).head(1)[['tema', 'texto']].reset_index(drop=True)
bow = CountVectorizer(lowercase=True, ngram_range=(1, 2), max_features=24)
X_bow = bow.fit_transform(microtextos['texto'])

tabla_bow = pd.DataFrame(
    X_bow.toarray(),
    columns=bow.get_feature_names_out(),
    index=microtextos['tema'],
)
display(microtextos)
display(tabla_bow)

### Decisiones de tokenización que cambian el análisis

- `lowercase=True` une “Inflación” e “inflación”.
- `ngram_range=(1, 2)` conserva palabras y pares como “tasa desempleo”.
- `min_df` elimina términos demasiado raros.
- `max_df` puede retirar términos presentes en casi todos los documentos.
- eliminar tildes, números o palabras vacías puede ayudar, pero también puede destruir señal.

**Advertencia:** limpiar más no equivale a modelar mejor. Por ejemplo, negar “no aumentó” y conservar solo “aumentó” puede invertir el sentido.

## 3. TF–IDF

Una versión usual de la ponderación es:

\[
\operatorname{tfidf}(t,d)=\operatorname{tf}(t,d)\times \log\left(\frac{1+N}{1+df(t)}\right).
\]

Un término recibe peso alto cuando aparece en el documento, pero no domina todo el corpus. TF–IDF no “entiende” economía; organiza patrones de frecuencia.

In [ ]:
tfidf_demo = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=0.95, max_features=80)
X_tfidf_demo = tfidf_demo.fit_transform(corpus['texto'])

print('Forma de la matriz:', X_tfidf_demo.shape)
print('Porcentaje de ceros: {:.2%}'.format(1 - X_tfidf_demo.nnz / np.prod(X_tfidf_demo.shape)))

doc_idx = 10
pesos = pd.Series(
    X_tfidf_demo[doc_idx].toarray().ravel(),
    index=tfidf_demo.get_feature_names_out(),
).sort_values(ascending=False)
display(corpus.loc[[doc_idx], ['texto_id', 'texto', 'tema']])
display(pesos.head(10).to_frame('peso_tfidf'))

### Ejercicio semi-asistido 1 · Comparar representaciones

Construya una tabla para dos textos elegidos por usted. Compare sus cinco términos con mayor conteo y sus cinco términos con mayor TF–IDF.

**Entrada esperada**

```python
ids = [12, 80]
```

**Forma de salida esperada**

| documento | término | conteo | tfidf |
|---|---|---:|---:|
| 12 | ... | ... | ... |

No se exige que los términos coincidan con un ejemplo predeterminado. Sí se exige explicar por qué los rankings difieren.

In [ ]:
# Plantilla semi-asistida: complete los pasos marcados.
ids = [12, 80]
textos_elegidos = corpus.loc[ids, 'texto']

# 1. Ajuste CountVectorizer y TfidfVectorizer sobre todo el corpus.
# 2. Transforme únicamente textos_elegidos.
# 3. Lleve ambas matrices a formato largo y una por documento/término.
# 4. Conserve los cinco mayores pesos de cada documento.

resultado_ej1 = pd.DataFrame(columns=['documento', 'término', 'conteo', 'tfidf'])
resultado_ej1

## 4. Embeddings densos con SVD

Una matriz TF–IDF suele tener miles de columnas y muchos ceros. `TruncatedSVD` busca unas pocas direcciones que resumen patrones de coocurrencia:

\[
X_{\text{tfidf}} \approx U_k\Sigma_kV_k^\top.
\]

El producto reducido funciona como un *embedding* del documento. Textos cercanos en ese espacio comparten patrones léxicos latentes, aunque no necesariamente el mismo significado económico.

In [ ]:
representacion_densa = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_df=0.92)),
    ('svd', TruncatedSVD(n_components=12, random_state=RANDOM_STATE)),
    ('normalizar', Normalizer(copy=False)),
])

X_emb = representacion_densa.fit_transform(corpus['texto'])
print('Dimensión TF–IDF:', representacion_densa.named_steps['tfidf'].transform(corpus['texto']).shape)
print('Dimensión del embedding:', X_emb.shape)
print('Varianza acumulada aproximada: {:.1%}'.format(
    representacion_densa.named_steps['svd'].explained_variance_ratio_.sum()
))

### Similitud coseno

Para vectores normalizados, la similitud coseno está entre −1 y 1 y compara dirección más que magnitud:

\[
\cos(\mathbf{x},\mathbf{z})=\frac{\mathbf{x}\cdot\mathbf{z}}{\|\mathbf{x}\|\|\mathbf{z}\|}.
\]

Una similitud alta es una señal para recuperar candidatos; **no demuestra que un texto respalde una afirmación**.

In [ ]:
def vecinos_textuales(indice, embeddings, datos, k=5):
    distancias = pairwise_distances(embeddings[indice:indice+1], embeddings, metric='cosine').ravel()
    orden = np.argsort(distancias)
    orden = orden[orden != indice][:k]
    salida = datos.loc[orden, ['texto_id', 'tema', 'sentimiento', 'texto']].copy()
    salida.insert(0, 'similitud', 1 - distancias[orden])
    return salida


consulta_idx = 25
display(corpus.loc[[consulta_idx], ['texto_id', 'tema', 'sentimiento', 'texto']])
display(vecinos_textuales(consulta_idx, X_emb, corpus, k=5))

### Pausa de interpretación

1. ¿Los vecinos comparten tema, sentimiento, ambos o ninguno?
2. ¿Qué expresiones parecen explicar la cercanía?
3. ¿Un vecino cercano puede contradecir al texto de consulta?
4. ¿Qué ventaja tendría un *embedding* preentrenado? ¿Qué nueva dependencia introduciría?

#### Extensión opcional: transformers

Si el entorno tiene `sentence-transformers` y acceso al modelo, puede comparar estos vectores con un modelo multilingüe. No es requisito del taller.

```python
# from sentence_transformers import SentenceTransformer
# modelo = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
# X_transformer = modelo.encode(corpus['texto'].tolist(), normalize_embeddings=True)
```

La comparación debe hacerse con la misma consulta y un criterio definido de antemano; no basta con afirmar que “transformer es mejor”.

## 5. Clasificación supervisada de temas

Tenemos una etiqueta conocida `tema`. El flujo correcto conserva el conjunto de prueba aislado y ajusta el vectorizador únicamente con los datos de entrenamiento.

```text
texto crudo → split estratificado → baseline → pipeline TF–IDF + modelo
           → validación cruzada en train → evaluación única en test
```

Poner TF–IDF dentro del `Pipeline` evita que el vocabulario e IDF aprendan de los pliegues de validación o del test.

In [ ]:
X = corpus['texto']
y = corpus['tema']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=y,
)

print('Train:', X_train.shape, '| Test:', X_test.shape)
display(pd.concat([
    y_train.value_counts(normalize=True).rename('train'),
    y_test.value_counts(normalize=True).rename('test'),
], axis=1).round(3))

### Baseline

El baseline responde: “¿qué obtendría con una regla mínima?”. Para múltiples clases usamos la categoría más frecuente. Como las clases no son perfectamente balanceadas, reportaremos `accuracy` y **F1 macro**, que promedia el F1 de cada clase con el mismo peso.

In [ ]:
baseline = DummyClassifier(strategy='most_frequent')
baseline.fit(X_train.to_frame(), y_train)
pred_base = baseline.predict(X_test.to_frame())

print('Accuracy baseline:', (pred_base == y_test).mean().round(3))
print('F1 macro baseline:', round(f1_score(y_test, pred_base, average='macro'), 3))

### Pipeline TF–IDF + regresión logística

La regresión logística produce puntajes por clase a partir de los pesos TF–IDF. Aunque la entrada contiene bigramas y miles de columnas, la regularización limita coeficientes extremos.

In [ ]:
modelo_tema = Pipeline([
    ('tfidf', TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.95,
        sublinear_tf=True,
    )),
    ('modelo', LogisticRegression(
        max_iter=1500,
        class_weight='balanced',
        random_state=RANDOM_STATE,
    )),
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {'accuracy': 'accuracy', 'f1_macro': 'f1_macro'}
cv_resultados = cross_validate(modelo_tema, X_train, y_train, cv=cv, scoring=scoring)

pd.DataFrame({
    'métrica': ['accuracy', 'f1_macro'],
    'media_cv': [cv_resultados['test_accuracy'].mean(), cv_resultados['test_f1_macro'].mean()],
    'desv_cv': [cv_resultados['test_accuracy'].std(), cv_resultados['test_f1_macro'].std()],
}).round(3)

### Evaluación final en test

Solo después de cerrar las decisiones principales ajustamos sobre todo `train` y evaluamos una vez en `test`.

In [ ]:
modelo_tema.fit(X_train, y_train)
pred_test = modelo_tema.predict(X_test)

print(classification_report(y_test, pred_test, digits=3))

fig, ax = plt.subplots(figsize=(8, 6))
ConfusionMatrixDisplay.from_predictions(
    y_test,
    pred_test,
    normalize='true',
    xticks_rotation=30,
    cmap='PuRd',
    ax=ax,
)
ax.set_title('Matriz de confusión normalizada por clase real')
plt.tight_layout()
plt.show()

### ¿Qué términos usa el modelo?

Los coeficientes permiten inspeccionar asociaciones entre términos y clases. No son efectos causales y pueden capturar decisiones del proceso de generación o etiquetado.

In [ ]:
def terminos_por_clase(pipeline, n=8):
    vocab = pipeline.named_steps['tfidf'].get_feature_names_out()
    modelo = pipeline.named_steps['modelo']
    filas = []
    for clase, coef in zip(modelo.classes_, modelo.coef_):
        for idx in np.argsort(coef)[-n:][::-1]:
            filas.append({'clase': clase, 'término': vocab[idx], 'coeficiente': coef[idx]})
    return pd.DataFrame(filas)


display(terminos_por_clase(modelo_tema, n=6))

### Ejercicio semi-asistido 2 · Análisis de errores

Construya una tabla con los errores del test y responda:

- ¿qué pares de clases se confunden más?;
- ¿el error parece razonable al leer el fragmento?;
- ¿hay textos cuya etiqueta admite discusión?;
- ¿qué cambiaría primero: representación, datos, etiqueta o modelo?

**Salida mínima esperada**

| texto | real | predicción | confianza_predicción |
|---|---|---|---:|
| … | actividad | empleo | 0.54 |

In [ ]:
proba_test = modelo_tema.predict_proba(X_test)
errores = pd.DataFrame({
    'texto': X_test.values,
    'real': y_test.values,
    'predicción': pred_test,
    'confianza_predicción': proba_test.max(axis=1),
})
errores = errores.query('real != predicción').sort_values('confianza_predicción', ascending=False)

# Lea al menos cinco errores. Si no hay suficientes, explique por qué el corpus puede ser demasiado fácil.
errores.head(8)

## Cierre de la sesión 1 · Exit ticket

Responda en máximo cinco líneas por punto:

1. ¿Qué aprende TF–IDF y qué aprende la regresión logística?
2. ¿Por qué el vectorizador debe estar dentro del pipeline?
3. ¿Qué significa que dos textos sean cercanos en el embedding?
4. Mencione una razón por la que el desempeño observado podría ser optimista.

# Sesión 2 · Sentimiento, temas y validación con fuentes

## 6. Sentimiento no es “positivo o negativo para todo el mundo”

En economía, un mismo hecho puede beneficiar a un actor y perjudicar a otro. Por eso debemos definir el objeto de la etiqueta.

En este cuaderno, `sentimiento` significa **tono explícito del fragmento sobre la evolución del indicador mencionado**. No mide bienestar social, aprobación política ni efecto causal.

Tres fuentes habituales de error:

- negación: “la inflación no aumentó”;
- alcance: “mejoró el empleo, pero cayó el salario real”;
- perspectiva: “subió la tasa de interés” puede ser restrictivo para crédito y favorable para ciertos ahorradores.

## 7. Baseline léxico de sentimiento

Una regla por palabras es transparente, pero ignora contexto. Sirve como baseline y como instrumento de diagnóstico.

In [ ]:
LEX_POS = {'mejoró', 'recuperación', 'favorable', 'avanzó'}
LEX_NEG = {'deterioró', 'caída', 'desfavorable', 'retrocedió'}


def sentimiento_lexico(texto):
    tokens = set(re.findall(r'\b\w+\b', texto.lower()))
    puntaje = len(tokens & LEX_POS) - len(tokens & LEX_NEG)
    if puntaje > 0:
        return 'positivo'
    if puntaje < 0:
        return 'negativo'
    return 'neutral'


pred_lex = corpus['texto'].map(sentimiento_lexico)
print(classification_report(corpus['sentimiento'], pred_lex, digits=3))
pd.crosstab(corpus['sentimiento'], pred_lex, rownames=['real'], colnames=['regla'])

### Discusión guiada

1. ¿La regla tiene acceso a las palabras usadas para construir el corpus?
2. ¿Por qué eso puede inflar su resultado?
3. ¿Cómo evaluaría una regla léxica en datos reales anotados por varias personas?
4. ¿Qué haría con desacuerdos entre anotadores?

**Idea clave:** antes de optimizar el modelo, audite cómo se produjo la “verdad” contra la que se evalúa.

## 8. Modelo supervisado de sentimiento

Repetimos el mismo contrato de validación. Esta vez el objetivo cambia, pero la separación y el pipeline siguen siendo necesarios.

In [ ]:
Xs_train, Xs_test, ys_train, ys_test = train_test_split(
    corpus['texto'],
    corpus['sentimiento'],
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=corpus['sentimiento'],
)

modelo_sentimiento = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
    ('modelo', LogisticRegression(max_iter=1500, class_weight='balanced', random_state=RANDOM_STATE)),
])

res_sent = cross_validate(
    modelo_sentimiento,
    Xs_train,
    ys_train,
    cv=cv,
    scoring={'f1_macro': 'f1_macro', 'accuracy': 'accuracy'},
)
display(pd.DataFrame(res_sent)[['test_f1_macro', 'test_accuracy']].agg(['mean', 'std']).round(3))

modelo_sentimiento.fit(Xs_train, ys_train)
pred_sent = modelo_sentimiento.predict(Xs_test)
print(classification_report(ys_test, pred_sent, digits=3))

### Ejercicio semi-asistido 3 · Pruebas de contraste

Evalúe el modelo con cuatro frases diseñadas para desafiarlo:

```python
pruebas = [
    "La inflación no aumentó durante el mes.",
    "El desempleo cayó, pero la informalidad aumentó.",
    "La deuda aumentó menos de lo previsto.",
    "El informe describe la actividad sin concluir si mejoró.",
]
```

Genere una tabla con `texto`, `predicción` y las tres probabilidades. Luego explique por qué una probabilidad alta no garantiza que la lectura sea correcta fuera del dominio de entrenamiento.

In [ ]:
pruebas = pd.Series([
    'La inflación no aumentó durante el mes.',
    'El desempleo cayó, pero la informalidad aumentó.',
    'La deuda aumentó menos de lo previsto.',
    'El informe describe la actividad sin concluir si mejoró.',
], name='texto')

# Complete con predict y predict_proba. Mantenga una columna por clase.
resultado_pruebas = pruebas.to_frame()
resultado_pruebas['predicción'] = 'PENDIENTE'
for clase in modelo_sentimiento.named_steps['modelo'].classes_:
    resultado_pruebas[f'p_{clase}'] = np.nan
resultado_pruebas

## 9. Descubrimiento no supervisado de temas

En clasificación conocemos las etiquetas. En *topic modeling* pedimos al algoritmo que encuentre grupos de palabras que tienden a aparecer juntos.

Con NMF aproximamos una matriz no negativa:

\[
X \approx W H,
\]

donde cada fila de `W` representa la mezcla de temas de un documento y cada fila de `H` contiene los términos característicos de un componente.

El algoritmo entrega **componentes**, no nombres. Nombrarlos es una interpretación humana que debe justificarse leyendo documentos representativos.

In [ ]:
vectorizador_temas = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=4,
    max_df=0.85,
    sublinear_tf=True,
)
X_topics = vectorizador_temas.fit_transform(corpus['texto'])

nmf = NMF(n_components=5, init='nndsvda', random_state=RANDOM_STATE, max_iter=500)
W = nmf.fit_transform(X_topics)
H = nmf.components_
vocab = vectorizador_temas.get_feature_names_out()

def palabras_por_componente(H, vocab, n=10):
    filas = []
    for k, pesos in enumerate(H):
        top = np.argsort(pesos)[-n:][::-1]
        filas.append({
            'componente': k,
            'palabras_clave': ', '.join(vocab[top]),
        })
    return pd.DataFrame(filas)

palabras_por_componente(H, vocab)

In [ ]:
componente_dominante = W.argmax(axis=1)
comparacion_temas = pd.crosstab(
    corpus['tema'],
    componente_dominante,
    normalize='index',
).round(2)
comparacion_temas

### Ejercicio semi-asistido 4 · Nombrar sin inventar

Para cada componente:

1. lea las palabras de mayor peso;
2. recupere tres documentos con mayor carga;
3. proponga un nombre corto;
4. cite palabras y documentos que justifican el nombre;
5. registre una alternativa plausible o una ambigüedad.

**Formato esperado**

| componente | nombre propuesto | evidencia léxica | documentos revisados | ambigüedad |
|---:|---|---|---|---|
| 0 | … | … | TX-…, TX-…, TX-… | … |

In [ ]:
def documentos_por_componente(k, W, datos, n=3):
    idx = np.argsort(W[:, k])[-n:][::-1]
    salida = datos.loc[idx, ['texto_id', 'tema', 'texto']].copy()
    salida.insert(1, 'peso_componente', W[idx, k])
    return salida


# Ejemplo de recuperación; cambie k entre 0 y 4.
documentos_por_componente(0, W, corpus, n=3)

## 10. Validación de resultados con fuentes textuales

Validar no significa buscar un fragmento “parecido” y dar por cierta una afirmación. Requiere al menos:

1. **trazabilidad:** identificar documento, fecha y emisor;
2. **correspondencia:** comprobar que el fragmento diga realmente lo afirmado;
3. **alcance:** revisar periodo, población, región y variable;
4. **contradicción:** buscar evidencia que limite o contradiga la lectura;
5. **incertidumbre:** distinguir descripción, predicción y causalidad.

Usaremos fuentes simuladas para aprender el procedimiento. La similitud coseno servirá para recuperar candidatos; la decisión final seguirá siendo humana.

In [ ]:
fuentes = pd.DataFrame([
    ['F-01', 'boletín estadístico', '2026-08-05', 'La inflación anual disminuyó en julio. La reducción se concentró en alimentos, mientras los servicios mostraron persistencia.'],
    ['F-02', 'informe laboral', '2026-08-12', 'La tasa de desempleo urbana cayó frente al año anterior. La mejora fue menor entre jóvenes y no permite concluir un efecto causal de una política específica.'],
    ['F-03', 'informe sectorial', '2026-08-18', 'La producción industrial retrocedió en junio, con heterogeneidad entre manufacturas. La construcción no forma parte de esta medición.'],
    ['F-04', 'reporte externo', '2026-08-20', 'Las exportaciones agrícolas aumentaron y las ventas manufactureras al exterior permanecieron estables. El informe no evalúa empleo.'],
    ['F-05', 'reporte fiscal', '2026-08-25', 'El recaudo tributario creció en el primer semestre. El gasto también aumentó, por lo que el documento no concluye que el déficit haya desaparecido.'],
    ['F-06', 'nota metodológica', '2026-08-28', 'Las cifras regionales no son directamente comparables sin ajustar cobertura, población y cambios en el marco muestral.'],
    ['F-07', 'boletín estadístico', '2026-09-01', 'La inflación básica permaneció estable. Las expectativas mostraron una reducción moderada, con amplia incertidumbre.'],
    ['F-08', 'informe laboral', '2026-09-03', 'La ocupación formal aumentó en zonas urbanas, pero la participación laboral también creció. Se recomienda analizar ambas tasas conjuntamente.'],
], columns=['fuente_id', 'tipo', 'fecha', 'fragmento'])
fuentes['fecha'] = pd.to_datetime(fuentes['fecha'])

afirmaciones = pd.DataFrame({
    'afirmacion_id': ['A-01', 'A-02', 'A-03', 'A-04'],
    'afirmacion': [
        'La inflación anual disminuyó por una reducción en alimentos.',
        'Una política pública causó la caída del desempleo juvenil.',
        'Las exportaciones agrícolas aumentaron.',
        'El déficit fiscal desapareció durante el primer semestre.',
    ],
})
display(fuentes)
display(afirmaciones)

### Recuperación de evidencia candidata

La función siguiente ajusta TF–IDF sobre fuentes y afirmaciones para encontrar los fragmentos más cercanos. La salida no es un veredicto automático.

**Ejemplo de entrada**

```python
recuperar_fuentes("Las exportaciones agrícolas aumentaron.", k=2)
```

**Forma de salida**

| similitud | fuente_id | fecha | fragmento |
|---:|---|---|---|
| 0.63 | F-04 | 2026-08-20 | … |

In [ ]:
def recuperar_fuentes(afirmacion, fuentes_df=fuentes, k=3):
    documentos = fuentes_df['fragmento'].tolist() + [afirmacion]
    vec = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)
    matriz = vec.fit_transform(documentos)
    sims = 1 - pairwise_distances(matriz[-1], matriz[:-1], metric='cosine').ravel()
    idx = np.argsort(sims)[-k:][::-1]
    salida = fuentes_df.iloc[idx][['fuente_id', 'tipo', 'fecha', 'fragmento']].copy()
    salida.insert(0, 'similitud', sims[idx])
    return salida.reset_index(drop=True)


recuperar_fuentes(afirmaciones.loc[2, 'afirmacion'], k=3)

### Matriz afirmación–evidencia

Clasifique cada afirmación después de leer las fuentes candidatas:

- **respaldada:** la fuente cubre variable, dirección, periodo y alcance;
- **parcial:** hay apoyo, pero la afirmación excede algún aspecto;
- **contradicha:** la fuente aporta evidencia incompatible;
- **sin evidencia suficiente:** no hay respaldo pertinente.

No use un umbral de similitud como sustituto de esta lectura.

In [ ]:
for afirmacion in afirmaciones.itertuples(index=False):
    print('\n', afirmacion.afirmacion_id, '-', afirmacion.afirmacion)
    display(recuperar_fuentes(afirmacion.afirmacion, k=2))

### Ejercicio semi-asistido 5 · Dictamen verificable

Complete la tabla. En `justificación`, cite el `fuente_id` y explique el ajuste o exceso de alcance.

In [ ]:
dictamen = afirmaciones.copy()
dictamen['estado'] = 'PENDIENTE'  # respaldada, parcial, contradicha o sin evidencia suficiente
dictamen['fuente_principal'] = 'PENDIENTE'
dictamen['justificación'] = 'PENDIENTE'
dictamen

## 11. Del resultado a una decisión prudente

Un tablero puede mostrar temas, sentimiento y volumen; una decisión exige contexto adicional. Antes de recomendar, pregunte:

- ¿qué error es más costoso: omitir una señal o producir una falsa alarma?;
- ¿las fuentes cubren el periodo y la población de interés?;
- ¿el modelo está fuera del dominio en el que fue validado?;
- ¿hay revisión humana de casos inciertos y afirmaciones materiales?;
- ¿la recomendación distingue predicción de explicación causal?

Una salida útil puede ser: “priorizar revisión humana de los documentos con alta probabilidad de tono negativo y evidencia contradictoria”. Una salida injustificada sería: “intervenir el sector porque el modelo demostró la causa de su deterioro”.

# Taller 4 · Monitor de coyuntura basado en texto

## Propósito

Construir un flujo reproducible que organice comunicaciones económicas, clasifique su tono, explore temas y valide afirmaciones contra fragmentos fuente.

**Modalidad:** autónoma, individual o en parejas según indique el docente.  
**Tiempo autónomo estimado:** 4–6 horas.  
**Entrega:** notebook ejecutado de principio a fin y una recomendación ejecutiva de máximo 250 palabras.

## Caso

Un equipo de análisis recibe comunicaciones de distinta procedencia. Tiene capacidad para revisar manualmente solo una fracción. Su tarea es construir un sistema de apoyo que:

1. describa el corpus;
2. clasifique `sentimiento`;
3. descubra temas sin usar `tema` durante el ajuste no supervisado;
4. recupere fuentes para verificar afirmaciones;
5. proponga una regla transparente de priorización para revisión humana.

El objetivo no es obtener una métrica perfecta. Es demostrar un proceso válido y reconocer los límites de los datos simulados.

## Datos del taller

El corpus se deriva reproduciblemente con otra semilla y se entrega en `taller`. La columna `tema` puede usarse **solo al final** para interpretar los componentes, no para entrenar NMF.

In [ ]:
taller = construir_corpus(n=520, semilla=2026)

# Se introduce un pequeño conjunto de casos difíciles sin cambiar la etiqueta original.
casos_dificiles = pd.DataFrame([
    ['TA-X01', '2026-08-30', 'nota de investigación', 'La inflación no aumentó, aunque los servicios siguieron presionando el índice.', 'inflación', 'neutral'],
    ['TA-X02', '2026-08-31', 'informe sectorial', 'La ocupación mejoró, pero el ingreso laboral real se deterioró.', 'empleo', 'neutral'],
    ['TA-X03', '2026-09-01', 'reporte fiscal', 'El recaudo cayó menos de lo previsto y la incertidumbre fiscal continúa.', 'fiscal', 'negativo'],
    ['TA-X04', '2026-09-02', 'reporte externo', 'Las exportaciones crecieron; sin embargo, la balanza comercial se deterioró.', 'sector externo', 'neutral'],
], columns=taller.columns)
casos_dificiles['fecha'] = pd.to_datetime(casos_dificiles['fecha'])
taller = pd.concat([taller, casos_dificiles], ignore_index=True)

print(taller.shape)
taller.sample(5, random_state=RANDOM_STATE)

## Pregunta orientadora y restricción de capacidad

> ¿Cómo puede el equipo priorizar el 15 % de los textos para revisión humana, combinando riesgo de tono negativo, incertidumbre del clasificador y necesidad de verificación?

La regla debe ser auditable. No puede usar la etiqueta real en producción y debe indicar qué ocurre con textos fuera de dominio.

## Fase 1 · Contrato analítico — 8 puntos

Defina antes de modelar:

- unidad de análisis;
- variable objetivo y significado operacional;
- población a la que aplica el sistema;
- momento en que estaría disponible cada dato;
- uso permitido de la predicción;
- dos usos que serían injustificados.

**Producto:** una tabla de seis filas `elemento | definición | riesgo si se interpreta mal`.

In [ ]:
# TODO Fase 1
contrato = pd.DataFrame(columns=['elemento', 'definición', 'riesgo si se interpreta mal'])
contrato

## Fase 2 · Auditoría del corpus — 10 puntos

Reporte tamaño, periodo, faltantes, duplicados, longitud de textos, distribución por fuente, tema y sentimiento. Identifique al menos dos señales de posible sesgo o artificialidad.

**Salida mínima esperada:** tabla de auditoría, dos gráficos legibles y un párrafo de interpretación.

In [ ]:
# TODO Fase 2
# Sugerencias: isna(), duplicated(), str.split().str.len(), value_counts(), crosstab().

## Fase 3 · Split y baseline — 10 puntos

1. Separe 75 % para entrenamiento y 25 % para prueba con estratificación por `sentimiento`.
2. Mantenga los cuatro casos `TA-X...` fuera del ajuste y úselos como prueba de contraste adicional.
3. Construya un `DummyClassifier`.
4. Reporte accuracy y F1 macro.

**Criterio:** la prueba permanece aislada hasta cerrar las decisiones de modelamiento.

In [ ]:
# TODO Fase 3

## Fase 4 · Dos representaciones y validación — 18 puntos

Compare mediante la misma validación cruzada:

- Modelo A: TF–IDF + regresión logística.
- Modelo B: TF–IDF + `TruncatedSVD` + normalización + regresión logística.

Use cinco pliegues estratificados y reporte media y desviación de F1 macro. Todas las transformaciones deben estar dentro de cada pipeline.

**Tabla de salida esperada**

| modelo | f1_macro_cv_media | f1_macro_cv_desv | tiempo_ajuste | dimensión |
|---|---:|---:|---:|---:|
| TF–IDF | … | … | … | … |
| Embedding SVD | … | … | … | … |

Seleccione un modelo y justifique con desempeño, costo e interpretabilidad.

In [ ]:
# TODO Fase 4
# Pista: para el modelo B, ubique TfidfVectorizer, TruncatedSVD,
# Normalizer y LogisticRegression en un solo Pipeline.

## Fase 5 · Evaluación final y errores — 14 puntos

Evalúe una sola vez el modelo seleccionado en test. Incluya:

- classification report;
- matriz de confusión absoluta y normalizada;
- comparación explícita con el baseline;
- lectura de ocho errores;
- resultados separados para los cuatro casos difíciles.

No describa `accuracy` como “porcentaje de verdad”. Es proporción de etiquetas acertadas bajo este conjunto y esta definición.

In [ ]:
# TODO Fase 5

## Fase 6 · Incertidumbre y priorización — 12 puntos

Para cada texto, construya:

- `p_negativo`;
- `confianza_maxima`;
- `incertidumbre = 1 - confianza_maxima`;
- `puntaje_revision`, definido y justificado por usted.

Priorice exactamente el 15 % con mayor puntaje. La regla debe poder calcularse sin conocer la etiqueta verdadera.

**Ejemplo de forma de salida**

| texto_id | p_negativo | incertidumbre | puntaje_revision | revisar |
|---|---:|---:|---:|---|
| TX-… | 0.71 | 0.22 | 0.60 | Sí |

Explique el costo económico de una falsa alarma y el de omitir un texto relevante.

In [ ]:
# TODO Fase 6

## Fase 7 · Descubrimiento de temas — 12 puntos

1. Ajuste NMF con entre 4 y 7 componentes usando solo `texto`.
2. Muestre diez términos y tres documentos representativos por componente.
3. Proponga nombres provisionales.
4. Solo después compare con `tema` para diagnosticar correspondencia y mezcla.
5. Discuta al menos un componente ambiguo.

No es válido nombrar un componente mirando primero la etiqueta real.

In [ ]:
# TODO Fase 7

## Fase 8 · Validación con fuentes — 10 puntos

Redacte tres afirmaciones derivadas de su análisis:

- una sobre sentimiento;
- una sobre un tema;
- una sobre priorización.

Para cada afirmación, recupere dos fuentes candidatas usando `recuperar_fuentes`, lea los fragmentos y asigne un estado: respaldada, parcial, contradicha o sin evidencia suficiente.

**Importante:** las fuentes simuladas son pequeñas y pueden no cubrir el taller completo. “Sin evidencia suficiente” es un resultado válido y, en ocasiones, el más responsable.

In [ ]:
# TODO Fase 8
matriz_evidencia = pd.DataFrame(columns=[
    'afirmación', 'fuentes_revisadas', 'estado', 'justificación', 'ajuste_necesario'
])
matriz_evidencia

## Fase 9 · Recomendación ejecutiva — 6 puntos

Escriba máximo 250 palabras para el equipo de análisis. Debe incluir:

1. qué puede hacer el sistema;
2. qué modelo seleccionó y contra qué baseline lo comparó;
3. cómo prioriza el 15 %;
4. un hallazgo respaldado;
5. dos limitaciones;
6. una decisión concreta de implementación o no implementación.

No use lenguaje causal salvo que exista un diseño causal, que no forma parte de este taller.

In [ ]:
# TODO Fase 9: escriba la recomendación en una celda Markdown.

## Fase 10 · Reproducibilidad — requisito transversal

Antes de entregar:

- reinicie el kernel y ejecute todas las celdas;
- confirme que no haya errores;
- conserve la semilla aleatoria;
- deje transformaciones dentro del pipeline;
- no seleccione el modelo ni ajuste reglas mirando el test;
- identifique claramente qué datos son simulados;
- cite cualquier fuente externa adicional;
- elimine salidas irrelevantes o excesivas.

## Rúbrica del Taller 4

| Criterio | Puntos |
|---|---:|
| Contrato analítico | 8 |
| Auditoría del corpus | 10 |
| Split y baseline | 10 |
| Representaciones y validación | 18 |
| Evaluación y análisis de errores | 14 |
| Incertidumbre y priorización | 12 |
| Descubrimiento de temas | 12 |
| Validación con fuentes | 10 |
| Recomendación ejecutiva | 6 |
| **Total** | **100** |

### Penalizaciones conceptuales

- fuga de información o ajuste del vectorizador antes del split: hasta −20;
- usar el test repetidamente para escoger el modelo: hasta −15;
- presentar similitud como prueba automática: hasta −10;
- afirmar causalidad sin diseño causal: hasta −10;
- ocultar que los datos son simulados: hasta −10.

# Lista de verificación final

- [ ] Definí qué significa cada etiqueta.
- [ ] Separé los datos antes de ajustar representaciones.
- [ ] Comparé contra un baseline.
- [ ] Usé validación cruzada solo dentro de entrenamiento.
- [ ] Reporté F1 macro y no solo accuracy.
- [ ] Leí errores y casos de contraste.
- [ ] Separé clasificación supervisada de temas no supervisados.
- [ ] Verifiqué afirmaciones leyendo fragmentos fuente.
- [ ] Expresé límites y no hice afirmaciones causales.
- [ ] El notebook corre de principio a fin.

## Cierre

El texto puede convertirse en variables, probabilidades y componentes. El rigor aparece cuando mantenemos unidas tres capas: **representación**, **validación predictiva** y **verificación documental**. Si se separan, un sistema técnicamente correcto puede producir una conclusión económicamente indefendible.